In [24]:
from dotenv import load_dotenv
load_dotenv() 

False

In [27]:
from dotenv import load_dotenv
import json
import requests
import pyodbc
from datetime import datetime
import os

load_dotenv()  # loads .env file
# API_KEY = os.getenv("OWM_API_KEY")
API_KEY = "04cd75cda4242fc40ad577a84ab9d38f"
conn_str = r"DRIVER={ODBC Driver 17 for SQL Server};SERVER=localhost\SQLEXPRESS;DATABASE=IOT;Trusted_Connection=yes;"


In [28]:
print(API_KEY)

04cd75cda4242fc40ad577a84ab9d38f


In [29]:
cities = [
    "Cairo", "Alexandria",
    "New York", "Los Angeles",
    "London", "Manchester",
    "Berlin", "Munich",
    "Paris", "Marseille",
    "Mumbai", "Delhi",
    "Tokyo", "Osaka",
    "Sao Paulo", "Rio de Janeiro",
    "Toronto", "Vancouver",
    "Sydney", "Melbourne"
]

In [30]:
conn = pyodbc.connect(conn_str)
cursor = conn.cursor()

for city_name in cities:
    url = f"http://api.openweathermap.org/data/2.5/weather?q={city_name}&appid={API_KEY}&units=metric"
    response = requests.get(url)
    if response.status_code != 200:
        print(f"Skipping {city_name}, API error: {response.status_code}")
        continue
    data = response.json()

    # Extract city info
    city_name_api = data.get("name")
    country = data.get("sys", {}).get("country")
    lat = data.get("coord", {}).get("lat")
    lon = data.get("coord", {}).get("lon")

    # Insert city if not exists
    cursor.execute("SELECT CityID FROM Cities WHERE CityName=? AND Country=?", city_name_api, country)
    row = cursor.fetchone()

    if row:
        city_id = row[0]
    else:
        cursor.execute(
            "INSERT INTO Cities (CityName, Country, Latitude, Longitude) VALUES (?, ?, ?, ?); SELECT SCOPE_IDENTITY();",
            city_name_api, country, lat, lon
        )
    conn.commit()

    # city_id = cursor.fetchone()[0]

    # Extract weather info
    main = data.get("main", {})
    wind = data.get("wind", {})
    weather_list = data.get("weather", [{}])

    temp = main.get("temp")
    feels_like = main.get("feels_like")
    humidity = main.get("humidity")
    pressure = main.get("pressure")
    wind_speed = wind.get("speed", 0)
    wind_deg = wind.get("deg", 0)
    weather_main = weather_list[0].get("main")
    weather_desc = weather_list[0].get("description")
    now = datetime.now()

    # Insert weather measurement
    cursor.execute("""
        INSERT INTO WeatherMeasurements
        (CityID, Temperature, FeelsLike, Humidity, Pressure, WindSpeed, WindDirection, WeatherMain, WeatherDescription, DateTimeRecorded)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, city_id, temp, feels_like, humidity, pressure, wind_speed, wind_deg, weather_main, weather_desc, now)
    conn.commit()

    print(f"Inserted data for {city_name_api}, {country}")

conn.close()
print("All cities processed successfully!")

Inserted data for Cairo, EG
Inserted data for Alexandria, EG
Inserted data for New York, US
Inserted data for Los Angeles, US
Inserted data for London, GB
Inserted data for Manchester, GB
Inserted data for Berlin, DE
Inserted data for Munich, DE
Inserted data for Paris, FR
Inserted data for Arrondissement de Marseille, FR
Inserted data for Mumbai, IN
Inserted data for Delhi, IN
Inserted data for Tokyo, JP
Inserted data for Osaka, JP
Inserted data for São Paulo, BR
Inserted data for Rio de Janeiro, BR
Inserted data for Toronto, CA
Inserted data for Vancouver, CA
Inserted data for Sydney, AU
Inserted data for Melbourne, US
All cities processed successfully!
